# Week 2, Day 4: Deep Research Application

## What this lab covers

This lab assembles a multi-agent deep-research workflow. The agents divide the work between web search, research planning, report writing, and email delivery, then coordinate their results into a researched response.

In [5]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel, set_tracing_disabled, function_tool, WebSearchTool, gen_trace_id
from openai import AsyncAzureOpenAI 
from anthropic import AsyncAnthropicFoundry
from agents.model_settings import ModelSettings
from IPython.display import Markdown, display
from agents.extensions.visualization import draw_graph
import asyncio
from typing import Dict, Any, List, Optional, Union
import datetime
import win32com.client
from pydantic import BaseModel, Field
import pythoncom
import subprocess
import re
import time
from messenger import send_email_tool, record_message_tool

set_tracing_disabled(True)  # Disable tracing for this notebook to avoid cluttering the output
load_dotenv(override=True) # Load environment variables from .env file, override existing ones if necessary

True

In [3]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")

# From Anthropic
AZURE_CLAUDE_DEPLOYMENT_OPUS_48=os.getenv("AZURE_CLAUDE_DEPLOYMENT_OPUS_48")
AZURE_CLAUDE_ENDPOINT=os.getenv("AZURE_CLAUDE_ENDPOINT")
AZURE_CLAUDE_API_KEY=os.getenv("AZURE_CLAUDE_API_KEY")
if AZURE_CLAUDE_API_KEY:
    print("AZURE_CLAUDE_API_KEY is avaiable")
else:
    print("AZURE_CLAUDE_API_KEY is not available")

# Loading Email addresses
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS_TO")
if EMAIL_ADDRESS: 
    print("Email address found!")
else:
    print("Email address not found!")

AZURE_OPENAI_API_KEY is available
AZURE_CLAUDE_API_KEY is avaiable
Email address found!


In [6]:
## setting up the client for OpenAI using Azure Endpoint (Azure Foundry)
client_openai = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)
client_anthropic = AsyncAnthropicFoundry(
    base_url=AZURE_CLAUDE_ENDPOINT,
    api_key=AZURE_CLAUDE_API_KEY
)


## let's a define a model pointing to Azure deployment
model_openai_GPT_54 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_54_mini
)
model_openai_GPT_41 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_41
)
model_openai_CLAUDE_OPUS_48 = OpenAIChatCompletionsModel(
    openai_client=client_anthropic,
    model=AZURE_CLAUDE_DEPLOYMENT_OPUS_48
)

### Strategy for the Deep Research Agent
 - We will build 4 Agents:
    1. The Search Agent: searches the web for information
    2. The Planner Agent: given a question, comes up with a list of searches that should be made.
    3. The Writer Agent: Writes a robust report
    4. The Emailer Agent: crafts and sends an email

#### Agent 01: The Search Agent

In [7]:
instructions = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a conise summary of the results. THe summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be brief & clear. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = []

In [ ]:
search_agent = Agent(
    name = "Search Agent",
    instructions=instructions,
    tools=tools,
    model=model_openai_GPT_54,
    model_settings=settings
)

result = await Runner.run(
    starting_agent = search_agent,
    input = task
)
print(f"Results of the search agent are as follows: \n{result.final_output}")

In [10]:
import requests

response = requests.get(
    "https://api.duckduckgo.com/",
    params={
        "q": "Most popular AI Agent frameworks in 2026",
        "format": "json"
    },
    verify = False
)

print(response.json())

{'Abstract': '', 'AbstractSource': '', 'AbstractText': '', 'AbstractURL': '', 'Answer': '', 'AnswerType': '', 'Definition': '', 'DefinitionSource': '', 'DefinitionURL': '', 'Entity': '', 'Heading': '', 'Image': '', 'ImageHeight': '', 'ImageIsLogo': '', 'ImageWidth': '', 'Infobox': '', 'Redirect': '', 'RelatedTopics': [], 'Results': [], 'Type': '', 'meta': {'attribution': None, 'blockgroup': None, 'created_date': '2021-03-24', 'description': 'testing', 'designer': None, 'dev_date': '2021-03-24', 'dev_milestone': 'development', 'developer': [{'name': 'zt', 'type': 'duck.co', 'url': 'https://duck.co/user/zt'}], 'example_query': '', 'id': 'just_another_test', 'is_stackexchange': 0, 'js_callback_name': 'another_test', 'live_date': None, 'maintainer': {'github': ''}, 'name': 'Just Another Test', 'perl_module': 'DDG::Lontail::AnotherTest', 'producer': None, 'production_state': 'offline', 'repo': 'fathead', 'signal_from': 'just_another_test', 'src_domain': 'how about there', 'src_id': None, 's

c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv\lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.duckduckgo.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
